In [144]:
import pandas as pd
import os
import re
import numpy as np

In [145]:
SCRIPT_DIR_PATH = os.getcwd()
CB_DIR_PATH = os.path.dirname(SCRIPT_DIR_PATH)
SSP_MODELING_DIR_PATH = os.path.dirname(CB_DIR_PATH)
TORNADO_DATA_DIR_PATH = os.path.join(SCRIPT_DIR_PATH, "data")
INPUT_DATA_DIR_PATH = os.path.join(TORNADO_DATA_DIR_PATH, "input/tornado")
OUTPUT_DATA_DIR_PATH = os.path.join(TORNADO_DATA_DIR_PATH, "output/tornado")

In [146]:
def add_sector_and_transformation_fields(df: pd.DataFrame, strategy_col: str = "strategy") -> pd.DataFrame:
    """
    Creates:
      - sector: sector code (e.g., AGRC)
      - transformation_name: transformation text after '... - <SECTOR>:'
    """
    df = df.copy()

    # --- sector extraction (captures 3-6 uppercase letters before colon) ---
    # Example: "Singleton - Default Value - AGRC: Improve rice..." -> AGRC
    df["sector"] = df[strategy_col].str.extract(r"-\s*([A-Z]{3,6})\s*:", expand=False)

    # Special case: baseline strategy
    df.loc[df[strategy_col].str.contains(r"^Strategy\s+TX:BASE", regex=True, na=False), "sector"] = "BASE"

    # --- transformation_name extraction ---
    # Keep only text after "<SECTOR>:"
    # Example -> "Improve rice..."
    df["transformation_name"] = df[strategy_col].str.extract(r":\s*(.*)$", expand=False)

    # If it's baseline, keep the full strategy string as the name (or label it as BASE)
    base_mask = df[strategy_col].str.contains(r"^Strategy\s+TX:BASE", regex=True, na=False)
    df.loc[base_mask, "transformation_name"] = "BASE"

    # Clean whitespace
    df["transformation_name"] = df["transformation_name"].fillna("").str.strip()

    return df

## Load and process emission data

In [147]:
# Load the decomposed emissions long format data
emissions_df = pd.read_csv(os.path.join(INPUT_DATA_DIR_PATH, "decomposed_emissions_libya_2022_tornado_data_raw.csv"))
emissions_df.head()

,strategy_id,primary_id,ID,sector,subsector,value,Year,Gas,design_id,future_id,strategy,Code,Contry,source
0,0.0,0.0,1.A.1 - Energy Industries:CH4,1 - Energy,1.A.1 - Energy Industries,0.035964,2023,CH4,0.0,0.0,Strategy TX:BASE,LBY,libya,SISEPUEDE
1,0.0,0.0,1.A.1 - Energy Industries:CH4,1 - Energy,1.A.1 - Energy Industries,0.037243,2024,CH4,0.0,0.0,Strategy TX:BASE,LBY,libya,SISEPUEDE
2,0.0,0.0,1.A.1 - Energy Industries:CH4,1 - Energy,1.A.1 - Energy Industries,0.038469,2025,CH4,0.0,0.0,Strategy TX:BASE,LBY,libya,SISEPUEDE
3,0.0,0.0,1.A.1 - Energy Industries:CH4,1 - Energy,1.A.1 - Energy Industries,0.039256,2026,CH4,0.0,0.0,Strategy TX:BASE,LBY,libya,SISEPUEDE
4,0.0,0.0,1.A.1 - Energy Industries:CH4,1 - Energy,1.A.1 - Energy Industries,0.039666,2027,CH4,0.0,0.0,Strategy TX:BASE,LBY,libya,SISEPUEDE


In [148]:
print(emissions_df.primary_id.nunique())

69


In [149]:
print(emissions_df.primary_id.nunique())

69


In [150]:
# check unique strategy
emissions_df['strategy'].unique()

array(['Strategy TX:BASE',
       'Singleton - Default Value - AGRC: Improve rice management',
       'Singleton - Default Value - AGRC: Decrease Exports',
       'Singleton - Default Value - AGRC: Reduce supply chain losses',
       'Singleton - Default Value - AGRC: Expand conservation agriculture',
       'Singleton - Default Value - AGRC: Improve crop productivity',
       'Singleton - Default Value - FRST: Increase Sequestration',
       'Singleton - Default Value - LNDU: Bound Classes',
       'Singleton - Default Value - LNDU: Decrease loss of land use classes',
       'Singleton - Default Value - LNDU: Stop deforestation',
       'Singleton - Default Value - LNDU: Expand sustainable grazing practices',
       'Singleton - Default Value - LNDU: Increase Reforestation',
       'Singleton - Default Value - LNDU: Expand silvopasture',
       'Singleton - Default Value - LNDU: Partial land use reallocation',
       'Singleton - Default Value - LSMM: Increase biogas capture at anaero

In [151]:
# Drop historical and tx:base from df
filtered_emissions_df = emissions_df.loc[~emissions_df['strategy'].isin(['Historical'])]
print(emissions_df['strategy'].nunique())
print(filtered_emissions_df['strategy'].nunique())

70
69


In [152]:
filtered_emissions_df["strategy"].unique()

array(['Strategy TX:BASE',
       'Singleton - Default Value - AGRC: Improve rice management',
       'Singleton - Default Value - AGRC: Decrease Exports',
       'Singleton - Default Value - AGRC: Reduce supply chain losses',
       'Singleton - Default Value - AGRC: Expand conservation agriculture',
       'Singleton - Default Value - AGRC: Improve crop productivity',
       'Singleton - Default Value - FRST: Increase Sequestration',
       'Singleton - Default Value - LNDU: Bound Classes',
       'Singleton - Default Value - LNDU: Decrease loss of land use classes',
       'Singleton - Default Value - LNDU: Stop deforestation',
       'Singleton - Default Value - LNDU: Expand sustainable grazing practices',
       'Singleton - Default Value - LNDU: Increase Reforestation',
       'Singleton - Default Value - LNDU: Expand silvopasture',
       'Singleton - Default Value - LNDU: Partial land use reallocation',
       'Singleton - Default Value - LSMM: Increase biogas capture at anaero

In [153]:
filtered_emissions_df.tail()

,strategy_id,primary_id,ID,sector,subsector,value,Year,Gas,design_id,future_id,strategy,Code,Contry,source
79207,6001.0,72072.0,5- CCSQ:N2O,5- CCSQ,5- CCSQ,0.0,2046,N2O,0.0,0.0,Singleton - Default Value - PFLO: Industrial c...,LBY,libya,SISEPUEDE
79208,6001.0,72072.0,5- CCSQ:N2O,5- CCSQ,5- CCSQ,0.0,2047,N2O,0.0,0.0,Singleton - Default Value - PFLO: Industrial c...,LBY,libya,SISEPUEDE
79209,6001.0,72072.0,5- CCSQ:N2O,5- CCSQ,5- CCSQ,0.0,2048,N2O,0.0,0.0,Singleton - Default Value - PFLO: Industrial c...,LBY,libya,SISEPUEDE
79210,6001.0,72072.0,5- CCSQ:N2O,5- CCSQ,5- CCSQ,0.0,2049,N2O,0.0,0.0,Singleton - Default Value - PFLO: Industrial c...,LBY,libya,SISEPUEDE
79211,6001.0,72072.0,5- CCSQ:N2O,5- CCSQ,5- CCSQ,0.0,2050,N2O,0.0,0.0,Singleton - Default Value - PFLO: Industrial c...,LBY,libya,SISEPUEDE


In [154]:
# Now concat the original base df and the filtered emissions df
tornado_emissions_df = filtered_emissions_df
tornado_emissions_df['strategy'].unique()

array(['Strategy TX:BASE',
       'Singleton - Default Value - AGRC: Improve rice management',
       'Singleton - Default Value - AGRC: Decrease Exports',
       'Singleton - Default Value - AGRC: Reduce supply chain losses',
       'Singleton - Default Value - AGRC: Expand conservation agriculture',
       'Singleton - Default Value - AGRC: Improve crop productivity',
       'Singleton - Default Value - FRST: Increase Sequestration',
       'Singleton - Default Value - LNDU: Bound Classes',
       'Singleton - Default Value - LNDU: Decrease loss of land use classes',
       'Singleton - Default Value - LNDU: Stop deforestation',
       'Singleton - Default Value - LNDU: Expand sustainable grazing practices',
       'Singleton - Default Value - LNDU: Increase Reforestation',
       'Singleton - Default Value - LNDU: Expand silvopasture',
       'Singleton - Default Value - LNDU: Partial land use reallocation',
       'Singleton - Default Value - LSMM: Increase biogas capture at anaero

In [155]:
tornado_emissions_df.head()

,strategy_id,primary_id,ID,sector,subsector,value,Year,Gas,design_id,future_id,strategy,Code,Contry,source
0,0.0,0.0,1.A.1 - Energy Industries:CH4,1 - Energy,1.A.1 - Energy Industries,0.035964,2023,CH4,0.0,0.0,Strategy TX:BASE,LBY,libya,SISEPUEDE
1,0.0,0.0,1.A.1 - Energy Industries:CH4,1 - Energy,1.A.1 - Energy Industries,0.037243,2024,CH4,0.0,0.0,Strategy TX:BASE,LBY,libya,SISEPUEDE
2,0.0,0.0,1.A.1 - Energy Industries:CH4,1 - Energy,1.A.1 - Energy Industries,0.038469,2025,CH4,0.0,0.0,Strategy TX:BASE,LBY,libya,SISEPUEDE
3,0.0,0.0,1.A.1 - Energy Industries:CH4,1 - Energy,1.A.1 - Energy Industries,0.039256,2026,CH4,0.0,0.0,Strategy TX:BASE,LBY,libya,SISEPUEDE
4,0.0,0.0,1.A.1 - Energy Industries:CH4,1 - Energy,1.A.1 - Energy Industries,0.039666,2027,CH4,0.0,0.0,Strategy TX:BASE,LBY,libya,SISEPUEDE


In [156]:
# Aggregate by strategy_id, primary_id and strategy, and sum value
tornado_emissions_agg_df = tornado_emissions_df.groupby(
    ['strategy_id', 'primary_id', 'strategy']
)['value'].sum().reset_index()

tornado_emissions_agg_df.head()


,strategy_id,primary_id,strategy,value
0,0.0,0.0,Strategy TX:BASE,2751.782000
1,1000.0,1001.0,Singleton - Default Value - AGRC: Improve rice...,2751.782000
2,1001.0,2002.0,Singleton - Default Value - AGRC: Decrease Exp...,2751.782000
3,1002.0,3003.0,Singleton - Default Value - AGRC: Reduce suppl...,2751.666513
4,1003.0,4004.0,Singleton - Default Value - AGRC: Expand conse...,2753.552329


In [157]:
tornado_emissions_agg_df.tail()

,strategy_id,primary_id,strategy,value
64,4003.0,67067.0,Singleton - Default Value - IPPU: Reduce Nitro...,2751.390786
65,4004.0,68068.0,Singleton - Default Value - IPPU: Reduce other...,2751.782000
66,4005.0,69069.0,Singleton - Default Value - IPPU: Reduce use o...,2751.782000
67,6000.0,71071.0,Singleton - Default Value - PFLO: Change diets,2746.790351
68,6001.0,72072.0,Singleton - Default Value - PFLO: Industrial c...,2724.346988


In [158]:
# check if strategy id nunique matches amount of rows
print(tornado_emissions_agg_df['strategy_id'].nunique())
print(tornado_emissions_agg_df.shape[0])

69
69


In [159]:
# rename value to emission_total
tornado_emissions_agg_df = tornado_emissions_agg_df.rename(columns={'value': 'emission_total'})

# create base_emission_total column by setting it to the strategy_id == 0 value
base_emission_total = tornado_emissions_agg_df.loc[tornado_emissions_agg_df['strategy_id'] == 0, 'emission_total'].values[0]
tornado_emissions_agg_df['base_emission_total'] = base_emission_total

# calculate emission difference column
tornado_emissions_agg_df['emission_diff'] =  tornado_emissions_agg_df['emission_total'] - tornado_emissions_agg_df['base_emission_total']

tornado_emissions_agg_df['emission_diff'] = tornado_emissions_agg_df['emission_diff'].round(1)

tornado_emissions_agg_df.head(40)

,strategy_id,primary_id,strategy,emission_total,base_emission_total,emission_diff
0,0.0,0.0,Strategy TX:BASE,2751.782000,2751.782,0.0
1,1000.0,1001.0,Singleton - Default Value - AGRC: Improve rice...,2751.782000,2751.782,0.0
2,1001.0,2002.0,Singleton - Default Value - AGRC: Decrease Exp...,2751.782000,2751.782,0.0
3,1002.0,3003.0,Singleton - Default Value - AGRC: Reduce suppl...,2751.666513,2751.782,-0.1
4,1003.0,4004.0,Singleton - Default Value - AGRC: Expand conse...,2753.552329,2751.782,1.8
5,1004.0,5005.0,Singleton - Default Value - AGRC: Improve crop...,2757.219557,2751.782,5.4
6,1005.0,6006.0,Singleton - Default Value - FRST: Increase Seq...,2751.782000,2751.782,0.0
7,1006.0,7007.0,Singleton - Default Value - LNDU: Bound Classes,2751.782000,2751.782,0.0
8,1007.0,8008.0,Singleton - Default Value - LNDU: Decrease los...,2751.782000,2751.782,-0.0
9,1008.0,9009.0,Singleton - Default Value - LNDU: Stop defores...,2751.747550,2751.782,-0.0


In [160]:
tornado_emissions_agg_extended_df = add_sector_and_transformation_fields(tornado_emissions_agg_df)
tornado_emissions_agg_extended_df.head()

,strategy_id,primary_id,strategy,emission_total,base_emission_total,emission_diff,sector,transformation_name
0,0.0,0.0,Strategy TX:BASE,2751.782000,2751.782,0.0,BASE,BASE
1,1000.0,1001.0,Singleton - Default Value - AGRC: Improve rice...,2751.782000,2751.782,0.0,AGRC,Improve rice management
2,1001.0,2002.0,Singleton - Default Value - AGRC: Decrease Exp...,2751.782000,2751.782,0.0,AGRC,Decrease Exports
3,1002.0,3003.0,Singleton - Default Value - AGRC: Reduce suppl...,2751.666513,2751.782,-0.1,AGRC,Reduce supply chain losses
4,1003.0,4004.0,Singleton - Default Value - AGRC: Expand conse...,2753.552329,2751.782,1.8,AGRC,Expand conservation agriculture


In [161]:
tornado_emissions_agg_extended_df.to_clipboard(index=False)

## Load and process CB data

In [162]:
# cb_raw_df = pd.read_csv(os.path.join(INPUT_DATA_DIR_PATH, "costs_benefits_sisepuede_results_sisepuede_run_2026-01-29T15;28;40.322709_tornado_raw.csv"))
cb_raw_df = pd.read_csv(os.path.join(INPUT_DATA_DIR_PATH, "cba_results_ssp_modeling_tornado.csv"))
cb_raw_df.head()

,strategy_code,future_id,region,time_period,difference_variable,variable_value_baseline,variable_value_pathway,difference_value,variable,value
0,AGRC:DEC_CH4_RICE,0.0,libya,7.0,yield_agrc_bevs_and_spices_tonne,0.0,0.0,0.0,cb:agrc:crop_value:crops_produced:bevs_and_spices,0.0
1,AGRC:DEC_CH4_RICE,0.0,libya,8.0,yield_agrc_bevs_and_spices_tonne,0.0,0.0,0.0,cb:agrc:crop_value:crops_produced:bevs_and_spices,0.0
2,AGRC:DEC_CH4_RICE,0.0,libya,9.0,yield_agrc_bevs_and_spices_tonne,0.0,0.0,0.0,cb:agrc:crop_value:crops_produced:bevs_and_spices,0.0
3,AGRC:DEC_CH4_RICE,0.0,libya,10.0,yield_agrc_bevs_and_spices_tonne,0.0,0.0,0.0,cb:agrc:crop_value:crops_produced:bevs_and_spices,0.0
4,AGRC:DEC_CH4_RICE,0.0,libya,11.0,yield_agrc_bevs_and_spices_tonne,0.0,0.0,0.0,cb:agrc:crop_value:crops_produced:bevs_and_spices,0.0


In [163]:
# --- Create a copy of the raw data ---
cb_data = cb_raw_df.copy()

# Split 'variable' into components: name, sector, cb_type, item_1, item_2
# (Assumes exactly 5 colon-separated parts; if there are more colons inside the last field,
# they will be kept in item_2 thanks to n=4)
cb_chars = cb_data["variable"].astype(str).str.split(":", n=4, expand=True)
cb_chars.columns = ["name", "sector", "cb_type", "item_1", "item_2"]
cb_data = pd.concat([cb_data, cb_chars], axis=1)
cb_data.head()

,strategy_code,future_id,region,time_period,difference_variable,variable_value_baseline,variable_value_pathway,difference_value,variable,value,name,sector,cb_type,item_1,item_2
0,AGRC:DEC_CH4_RICE,0.0,libya,7.0,yield_agrc_bevs_and_spices_tonne,0.0,0.0,0.0,cb:agrc:crop_value:crops_produced:bevs_and_spices,0.0,cb,agrc,crop_value,crops_produced,bevs_and_spices
1,AGRC:DEC_CH4_RICE,0.0,libya,8.0,yield_agrc_bevs_and_spices_tonne,0.0,0.0,0.0,cb:agrc:crop_value:crops_produced:bevs_and_spices,0.0,cb,agrc,crop_value,crops_produced,bevs_and_spices
2,AGRC:DEC_CH4_RICE,0.0,libya,9.0,yield_agrc_bevs_and_spices_tonne,0.0,0.0,0.0,cb:agrc:crop_value:crops_produced:bevs_and_spices,0.0,cb,agrc,crop_value,crops_produced,bevs_and_spices
3,AGRC:DEC_CH4_RICE,0.0,libya,10.0,yield_agrc_bevs_and_spices_tonne,0.0,0.0,0.0,cb:agrc:crop_value:crops_produced:bevs_and_spices,0.0,cb,agrc,crop_value,crops_produced,bevs_and_spices
4,AGRC:DEC_CH4_RICE,0.0,libya,11.0,yield_agrc_bevs_and_spices_tonne,0.0,0.0,0.0,cb:agrc:crop_value:crops_produced:bevs_and_spices,0.0,cb,agrc,crop_value,crops_produced,bevs_and_spices


In [164]:
# Scale value from USD to billions (divide by 1e9)
if "value" in cb_data.columns:
    cb_data["value"] = cb_data["value"] / 1e9

# --- Remove "shifted" entries ---
# # Remove rows where item_2 contains "shifted"
# cb_data = cb_data[~cb_data["item_2"].astype(str).str.contains("shifted", na=False)]

# # Remove any remaining rows where variable contains "shifted2"
# cb_data = cb_data[~cb_data["variable"].astype(str).str.contains("shifted2", na=False)]

# --- Add Year column (Year = time_period + 2015) ---
cb_data["Year"] = cb_data["time_period"] + 2015

cb_data.head()

,strategy_code,future_id,region,time_period,difference_variable,variable_value_baseline,variable_value_pathway,difference_value,variable,value,name,sector,cb_type,item_1,item_2,Year
0,AGRC:DEC_CH4_RICE,0.0,libya,7.0,yield_agrc_bevs_and_spices_tonne,0.0,0.0,0.0,cb:agrc:crop_value:crops_produced:bevs_and_spices,0.0,cb,agrc,crop_value,crops_produced,bevs_and_spices,2022.0
1,AGRC:DEC_CH4_RICE,0.0,libya,8.0,yield_agrc_bevs_and_spices_tonne,0.0,0.0,0.0,cb:agrc:crop_value:crops_produced:bevs_and_spices,0.0,cb,agrc,crop_value,crops_produced,bevs_and_spices,2023.0
2,AGRC:DEC_CH4_RICE,0.0,libya,9.0,yield_agrc_bevs_and_spices_tonne,0.0,0.0,0.0,cb:agrc:crop_value:crops_produced:bevs_and_spices,0.0,cb,agrc,crop_value,crops_produced,bevs_and_spices,2024.0
3,AGRC:DEC_CH4_RICE,0.0,libya,10.0,yield_agrc_bevs_and_spices_tonne,0.0,0.0,0.0,cb:agrc:crop_value:crops_produced:bevs_and_spices,0.0,cb,agrc,crop_value,crops_produced,bevs_and_spices,2025.0
4,AGRC:DEC_CH4_RICE,0.0,libya,11.0,yield_agrc_bevs_and_spices_tonne,0.0,0.0,0.0,cb:agrc:crop_value:crops_produced:bevs_and_spices,0.0,cb,agrc,crop_value,crops_produced,bevs_and_spices,2026.0


In [165]:
# Load attribute strategy
attribute_strategy_df = pd.read_csv(os.path.join(INPUT_DATA_DIR_PATH, "ATTRIBUTE_STRATEGY.csv"))
attribute_strategy_df = attribute_strategy_df[["strategy_id", "strategy_code"]]
attribute_strategy_df.head()

,strategy_id,strategy_code
0,0,BASE
1,1000,AGRC:DEC_CH4_RICE
2,1001,AGRC:DEC_EXPORTS
3,1002,AGRC:DEC_LOSSES_SUPPLY_CHAIN
4,1003,AGRC:INC_CONSERVATION_AGRICULTURE


In [166]:
attribute_strategy_df.strategy_id.unique()

array([   0, 1000, 1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008, 1009,
       1010, 1011, 1012, 1013, 1014, 1015, 1016, 1017, 1018, 1019, 1020,
       1021, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009,
       2010, 2011, 3000, 3001, 3002, 3003, 3004, 3005, 3006, 3007, 3008,
       3009, 3010, 3011, 3012, 3013, 3014, 3015, 3016, 3017, 3018, 3019,
       3020, 3021, 3022, 3023, 3024, 3025, 4000, 4001, 4002, 4003, 4004,
       4005, 6000, 6001])

In [167]:
attribute_strategy_df.strategy_code.unique()

array(['BASE', 'AGRC:DEC_CH4_RICE', 'AGRC:DEC_EXPORTS',
       'AGRC:DEC_LOSSES_SUPPLY_CHAIN',
       'AGRC:INC_CONSERVATION_AGRICULTURE', 'AGRC:INC_PRODUCTIVITY',
       'FRST:INCREASE_SEQUESTRATION', 'LNDU:BOUND_CLASSES',
       'LNDU:DEC_CLASS_LOSS', 'LNDU:DEC_DEFORESTATION',
       'LNDU:DEC_SOC_LOSS_PASTURES', 'LNDU:INC_REFORESTATION',
       'LNDU:INC_SILVOPASTURE', 'LNDU:PLUR', 'LSMM:INC_CAPTURE_BIOGAS',
       'LSMM:INC_MANAGEMENT_CATTLE_PIGS', 'LSMM:INC_MANAGEMENT_OTHER',
       'LSMM:INC_MANAGEMENT_POULTRY', 'LVST:DEC_ENTERIC_FERMENTATION',
       'LVST:DEC_EXPORTS', 'LVST:INC_PRODUCTIVITY',
       'SOIL:DEC_LIME_APPLIED', 'SOIL:DEC_N_APPLIED',
       'TRWW:INC_CAPTURE_BIOGAS', 'TRWW:INC_COMPLIANCE_SEPTIC',
       'WALI:INC_TREATMENT_INDUSTRIAL', 'WALI:INC_TREATMENT_RURAL',
       'WALI:INC_TREATMENT_URBAN', 'WASO:DEC_CONSUMER_FOOD_WASTE',
       'WASO:INC_ANAEROBIC_AND_COMPOST', 'WASO:INC_CAPTURE_BIOGAS',
       'WASO:INC_ENERGY_FROM_BIOGAS', 'WASO:INC_ENERGY_FROM_INCINERATI

In [168]:
# Merge with cb_data on strategy_code
cb_data = cb_data.merge(attribute_strategy_df, on="strategy_code", how="left")
cb_data.head()

,strategy_code,future_id,region,time_period,difference_variable,variable_value_baseline,variable_value_pathway,difference_value,variable,value,name,sector,cb_type,item_1,item_2,Year,strategy_id
0,AGRC:DEC_CH4_RICE,0.0,libya,7.0,yield_agrc_bevs_and_spices_tonne,0.0,0.0,0.0,cb:agrc:crop_value:crops_produced:bevs_and_spices,0.0,cb,agrc,crop_value,crops_produced,bevs_and_spices,2022.0,1000
1,AGRC:DEC_CH4_RICE,0.0,libya,8.0,yield_agrc_bevs_and_spices_tonne,0.0,0.0,0.0,cb:agrc:crop_value:crops_produced:bevs_and_spices,0.0,cb,agrc,crop_value,crops_produced,bevs_and_spices,2023.0,1000
2,AGRC:DEC_CH4_RICE,0.0,libya,9.0,yield_agrc_bevs_and_spices_tonne,0.0,0.0,0.0,cb:agrc:crop_value:crops_produced:bevs_and_spices,0.0,cb,agrc,crop_value,crops_produced,bevs_and_spices,2024.0,1000
3,AGRC:DEC_CH4_RICE,0.0,libya,10.0,yield_agrc_bevs_and_spices_tonne,0.0,0.0,0.0,cb:agrc:crop_value:crops_produced:bevs_and_spices,0.0,cb,agrc,crop_value,crops_produced,bevs_and_spices,2025.0,1000
4,AGRC:DEC_CH4_RICE,0.0,libya,11.0,yield_agrc_bevs_and_spices_tonne,0.0,0.0,0.0,cb:agrc:crop_value:crops_produced:bevs_and_spices,0.0,cb,agrc,crop_value,crops_produced,bevs_and_spices,2026.0,1000


In [169]:
cb_data.strategy_code.unique()

array(['AGRC:DEC_CH4_RICE', 'AGRC:DEC_EXPORTS',
       'AGRC:DEC_LOSSES_SUPPLY_CHAIN',
       'AGRC:INC_CONSERVATION_AGRICULTURE', 'AGRC:INC_PRODUCTIVITY',
       'FRST:INCREASE_SEQUESTRATION', 'LNDU:BOUND_CLASSES',
       'LNDU:DEC_CLASS_LOSS', 'LNDU:DEC_DEFORESTATION',
       'LNDU:DEC_SOC_LOSS_PASTURES', 'LNDU:INC_REFORESTATION',
       'LNDU:INC_SILVOPASTURE', 'LNDU:PLUR', 'LSMM:INC_CAPTURE_BIOGAS',
       'LSMM:INC_MANAGEMENT_CATTLE_PIGS', 'LSMM:INC_MANAGEMENT_OTHER',
       'LSMM:INC_MANAGEMENT_POULTRY', 'LVST:DEC_ENTERIC_FERMENTATION',
       'LVST:DEC_EXPORTS', 'LVST:INC_PRODUCTIVITY',
       'SOIL:DEC_LIME_APPLIED', 'SOIL:DEC_N_APPLIED',
       'TRWW:INC_CAPTURE_BIOGAS', 'TRWW:INC_COMPLIANCE_SEPTIC',
       'WALI:INC_TREATMENT_INDUSTRIAL', 'WALI:INC_TREATMENT_RURAL',
       'WALI:INC_TREATMENT_URBAN', 'WASO:DEC_CONSUMER_FOOD_WASTE',
       'WASO:INC_ANAEROBIC_AND_COMPOST', 'WASO:INC_CAPTURE_BIOGAS',
       'WASO:INC_ENERGY_FROM_BIOGAS', 'WASO:INC_ENERGY_FROM_INCINERATION',
   

In [170]:
cb_data.strategy_id.unique()

array([1000, 1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008, 1009, 1010,
       1011, 1012, 1013, 1014, 1015, 1016, 1017, 1018, 1019, 1020, 1021,
       2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010,
       2011, 3000, 3001, 3002, 3003, 3004, 3005, 3006, 3007, 3008, 3009,
       3010, 3011, 3012, 3013, 3014, 3015, 3016, 3017, 3018, 3019, 3020,
       3021, 3022, 3023, 3024, 3025, 4000, 4001, 4002, 4003, 4004, 4005,
       6000, 6001])

In [171]:
# check for nans in strategy_id
cb_data[cb_data['strategy_id'].isna()]['strategy_code'].unique()

array([], dtype=object)

In [172]:
cb_data["sector"].unique()

array(['agrc', 'ccsq', 'entc', 'inen', 'ippu', 'lndu', 'lsmm', 'lvst',
       'scoe', 'soil', 'trns', 'trww', 'wali', 'waso', 'fgtv', 'pflo'],
      dtype=object)

In [173]:
cb_data.head()

,strategy_code,future_id,region,time_period,difference_variable,variable_value_baseline,variable_value_pathway,difference_value,variable,value,name,sector,cb_type,item_1,item_2,Year,strategy_id
0,AGRC:DEC_CH4_RICE,0.0,libya,7.0,yield_agrc_bevs_and_spices_tonne,0.0,0.0,0.0,cb:agrc:crop_value:crops_produced:bevs_and_spices,0.0,cb,agrc,crop_value,crops_produced,bevs_and_spices,2022.0,1000
1,AGRC:DEC_CH4_RICE,0.0,libya,8.0,yield_agrc_bevs_and_spices_tonne,0.0,0.0,0.0,cb:agrc:crop_value:crops_produced:bevs_and_spices,0.0,cb,agrc,crop_value,crops_produced,bevs_and_spices,2023.0,1000
2,AGRC:DEC_CH4_RICE,0.0,libya,9.0,yield_agrc_bevs_and_spices_tonne,0.0,0.0,0.0,cb:agrc:crop_value:crops_produced:bevs_and_spices,0.0,cb,agrc,crop_value,crops_produced,bevs_and_spices,2024.0,1000
3,AGRC:DEC_CH4_RICE,0.0,libya,10.0,yield_agrc_bevs_and_spices_tonne,0.0,0.0,0.0,cb:agrc:crop_value:crops_produced:bevs_and_spices,0.0,cb,agrc,crop_value,crops_produced,bevs_and_spices,2025.0,1000
4,AGRC:DEC_CH4_RICE,0.0,libya,11.0,yield_agrc_bevs_and_spices_tonne,0.0,0.0,0.0,cb:agrc:crop_value:crops_produced:bevs_and_spices,0.0,cb,agrc,crop_value,crops_produced,bevs_and_spices,2026.0,1000


In [174]:
# filter sectors
# target_sectors = ["wali", "trww", "waso", "soil", "ippu", "lvst", "agrc", "lndu", "lsmm"]
# cb_data = cb_data[cb_data["sector"].isin(target_sectors)].copy()
cb_data.head()

,strategy_code,future_id,region,time_period,difference_variable,variable_value_baseline,variable_value_pathway,difference_value,variable,value,name,sector,cb_type,item_1,item_2,Year,strategy_id
0,AGRC:DEC_CH4_RICE,0.0,libya,7.0,yield_agrc_bevs_and_spices_tonne,0.0,0.0,0.0,cb:agrc:crop_value:crops_produced:bevs_and_spices,0.0,cb,agrc,crop_value,crops_produced,bevs_and_spices,2022.0,1000
1,AGRC:DEC_CH4_RICE,0.0,libya,8.0,yield_agrc_bevs_and_spices_tonne,0.0,0.0,0.0,cb:agrc:crop_value:crops_produced:bevs_and_spices,0.0,cb,agrc,crop_value,crops_produced,bevs_and_spices,2023.0,1000
2,AGRC:DEC_CH4_RICE,0.0,libya,9.0,yield_agrc_bevs_and_spices_tonne,0.0,0.0,0.0,cb:agrc:crop_value:crops_produced:bevs_and_spices,0.0,cb,agrc,crop_value,crops_produced,bevs_and_spices,2024.0,1000
3,AGRC:DEC_CH4_RICE,0.0,libya,10.0,yield_agrc_bevs_and_spices_tonne,0.0,0.0,0.0,cb:agrc:crop_value:crops_produced:bevs_and_spices,0.0,cb,agrc,crop_value,crops_produced,bevs_and_spices,2025.0,1000
4,AGRC:DEC_CH4_RICE,0.0,libya,11.0,yield_agrc_bevs_and_spices_tonne,0.0,0.0,0.0,cb:agrc:crop_value:crops_produced:bevs_and_spices,0.0,cb,agrc,crop_value,crops_produced,bevs_and_spices,2026.0,1000


In [175]:
cb_data['sector'].unique()

array(['agrc', 'ccsq', 'entc', 'inen', 'ippu', 'lndu', 'lsmm', 'lvst',
       'scoe', 'soil', 'trns', 'trww', 'wali', 'waso', 'fgtv', 'pflo'],
      dtype=object)

In [176]:
# aggregate sum(value) grouped by strategy_id and cb_type
cb_data = (
    cb_data.groupby(["strategy_id", "cb_type"], as_index=False)["value"]
      .sum()
      .rename(columns={"value": "cumulative"})
)
cb_data.head()

,strategy_id,cb_type,cumulative
0,1000,air_pollution,0.0
1,1000,congestion,0.0
2,1000,crop_value,0.0
3,1000,ecosystem_services,0.0
4,1000,env_pollution,0.0


In [177]:
# unique cb_data types
cb_cats = cb_data["cb_type"].unique().tolist()

# long -> wide (R dcast equivalent)
wide_cb = (
    cb_data.pivot(index="strategy_id", columns="cb_type", values="cumulative")
      .reset_index()
)

# optional: remove column name from pivot for nicer printing
wide_cb.columns.name = None
wide_cb.head()

,strategy_id,air_pollution,congestion,consumer_savings,crop_value,ecosystem_services,env_pollution,fuel_cost,human_health,ippu_value,land_pollution,lvst_value,road_safety,sector_specific,system_cost,technical_cost,technical_savings,water_pollution
0,1000,0.000000,0.0,NaN,0.000000e+00,0.0,0.000000,0.00000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,NaN,0.0
1,1001,0.000000,0.0,NaN,0.000000e+00,0.0,0.000000,0.00000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,NaN,0.0
2,1002,-0.000116,0.0,NaN,3.073364e-15,0.0,0.086168,-0.00024,0.0,0.0,0.0,0.0,0.0,0.000474,0.0,0.236728,3.570858,0.0
3,1003,0.000000,0.0,25.369679,0.000000e+00,0.0,0.000000,0.00000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,2.971877,0.0
4,1004,-0.266912,0.0,NaN,4.168221e+01,0.0,0.000000,0.00000,0.0,0.0,0.0,0.0,0.0,-3.501832,0.0,-0.201055,NaN,0.0


In [178]:
cb_cats

['air_pollution',
 'congestion',
 'crop_value',
 'ecosystem_services',
 'env_pollution',
 'fuel_cost',
 'human_health',
 'ippu_value',
 'land_pollution',
 'lvst_value',
 'road_safety',
 'sector_specific',
 'system_cost',
 'technical_cost',
 'water_pollution',
 'technical_savings',
 'consumer_savings']

In [179]:
# --- 1) net_benefit = rowSums over all cb categories ---
wide_cb["net_benefit"] = wide_cb[cb_cats].sum(axis=1, skipna=True)

# --- 2) additional_benefits = rowSums excluding "technical_cost" ---
benefit_cols = [c for c in cb_cats if c != "technical_cost"]
wide_cb["additional_benefits"] = wide_cb[benefit_cols].sum(axis=1, skipna=True)

# --- 3) total_transformation_costs = rowSums over specific cols ---
cost_cols = ["technical_cost", "technical_savings", "fuel_cost"]

# (safe version: only use cols that exist in the df)
cost_cols = [c for c in cost_cols if c in wide_cb.columns]

wide_cb["total_transformation_costs"] = wide_cb[cost_cols].sum(axis=1, skipna=True)
wide_cb.head()

,strategy_id,air_pollution,congestion,consumer_savings,crop_value,ecosystem_services,env_pollution,fuel_cost,human_health,ippu_value,...,lvst_value,road_safety,sector_specific,system_cost,technical_cost,technical_savings,water_pollution,net_benefit,additional_benefits,total_transformation_costs
0,1000,0.000000,0.0,NaN,0.000000e+00,0.0,0.000000,0.00000,0.0,0.0,...,0.0,0.0,0.000000,0.0,0.000000,NaN,0.0,0.000000,0.000000,0.000000
1,1001,0.000000,0.0,NaN,0.000000e+00,0.0,0.000000,0.00000,0.0,0.0,...,0.0,0.0,0.000000,0.0,0.000000,NaN,0.0,0.000000,0.000000,0.000000
2,1002,-0.000116,0.0,NaN,3.073364e-15,0.0,0.086168,-0.00024,0.0,0.0,...,0.0,0.0,0.000474,0.0,0.236728,3.570858,0.0,3.893872,3.657145,3.807346
3,1003,0.000000,0.0,25.369679,0.000000e+00,0.0,0.000000,0.00000,0.0,0.0,...,0.0,0.0,0.000000,0.0,0.000000,2.971877,0.0,28.341555,28.341555,2.971877
4,1004,-0.266912,0.0,NaN,4.168221e+01,0.0,0.000000,0.00000,0.0,0.0,...,0.0,0.0,-3.501832,0.0,-0.201055,NaN,0.0,37.712410,37.913465,-0.201055


## Merge emissions and cb data and save

In [180]:
tornado_emissions_agg_extended_df[tornado_emissions_agg_extended_df.strategy == "Singleton - Default Value - AGRC: Improve rice management"]

,strategy_id,primary_id,strategy,emission_total,base_emission_total,emission_diff,sector,transformation_name
1,1000.0,1001.0,Singleton - Default Value - AGRC: Improve rice...,2751.782,2751.782,0.0,AGRC,Improve rice management


In [181]:
wide_cb.strategy_id.unique()

array([1000, 1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008, 1009, 1010,
       1011, 1012, 1013, 1014, 1015, 1016, 1017, 1018, 1019, 1020, 1021,
       2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010,
       2011, 3000, 3001, 3002, 3003, 3004, 3005, 3006, 3007, 3008, 3009,
       3010, 3011, 3012, 3013, 3014, 3015, 3016, 3017, 3018, 3019, 3020,
       3021, 3022, 3023, 3024, 3025, 4000, 4001, 4002, 4003, 4004, 4005,
       6000, 6001])

In [182]:
print(wide_cb.shape)
print(tornado_emissions_agg_extended_df.shape)

(68, 21)
(69, 8)


In [183]:
df_merged = pd.merge(
    tornado_emissions_agg_extended_df,
    wide_cb,
    on="strategy_id",
    how="inner"
)

df_merged.head()

,strategy_id,primary_id,strategy,emission_total,base_emission_total,emission_diff,sector,transformation_name,air_pollution,congestion,...,lvst_value,road_safety,sector_specific,system_cost,technical_cost,technical_savings,water_pollution,net_benefit,additional_benefits,total_transformation_costs
0,1000.0,1001.0,Singleton - Default Value - AGRC: Improve rice...,2751.782000,2751.782,0.0,AGRC,Improve rice management,0.000000,0.0,...,0.0,0.0,0.000000,0.0,0.000000,NaN,0.0,0.000000,0.000000,0.000000
1,1001.0,2002.0,Singleton - Default Value - AGRC: Decrease Exp...,2751.782000,2751.782,0.0,AGRC,Decrease Exports,0.000000,0.0,...,0.0,0.0,0.000000,0.0,0.000000,NaN,0.0,0.000000,0.000000,0.000000
2,1002.0,3003.0,Singleton - Default Value - AGRC: Reduce suppl...,2751.666513,2751.782,-0.1,AGRC,Reduce supply chain losses,-0.000116,0.0,...,0.0,0.0,0.000474,0.0,0.236728,3.570858,0.0,3.893872,3.657145,3.807346
3,1003.0,4004.0,Singleton - Default Value - AGRC: Expand conse...,2753.552329,2751.782,1.8,AGRC,Expand conservation agriculture,0.000000,0.0,...,0.0,0.0,0.000000,0.0,0.000000,2.971877,0.0,28.341555,28.341555,2.971877
4,1004.0,5005.0,Singleton - Default Value - AGRC: Improve crop...,2757.219557,2751.782,5.4,AGRC,Improve crop productivity,-0.266912,0.0,...,0.0,0.0,-3.501832,0.0,-0.201055,NaN,0.0,37.712410,37.913465,-0.201055


In [184]:
print(df_merged.shape)

(68, 28)


### Below we have some hardcoded fixed exclusive of this study case to replace incorrect tranformation names

In [185]:
df_merged.columns

Index(['strategy_id', 'primary_id', 'strategy', 'emission_total',
       'base_emission_total', 'emission_diff', 'sector', 'transformation_name',
       'air_pollution', 'congestion', 'consumer_savings', 'crop_value',
       'ecosystem_services', 'env_pollution', 'fuel_cost', 'human_health',
       'ippu_value', 'land_pollution', 'lvst_value', 'road_safety',
       'sector_specific', 'system_cost', 'technical_cost', 'technical_savings',
       'water_pollution', 'net_benefit', 'additional_benefits',
       'total_transformation_costs'],
      dtype='object')

In [186]:
# multiply technical_cost by -1 to get positive costs
df_merged['technical_cost'] = df_merged['technical_cost'] * -1

# create marginal total abatement cost column
df_merged['marginal_total_abatement_cost_(USD/tCO2e)'] = (df_merged['technical_cost'] / df_merged['emission_diff'])*1000

# If technical_cost is positive then marginal_total_abatement_cost should be positive too.
# df_merged["marginal_total_abatement_cost_(USD/tCO2e)"] = np.where(df_merged["technical_cost"] > 0, df_merged["marginal_total_abatement_cost_(USD/tCO2e)"].abs(), df_merged["marginal_total_abatement_cost_(USD/tCO2e)"])
df_merged["marginal_total_abatement_cost_(USD/tCO2e)"] = df_merged["marginal_total_abatement_cost_(USD/tCO2e)"].abs() * np.sign(df_merged["technical_cost"])


In [187]:
df_merged["strategy"].unique()

array(['Singleton - Default Value - AGRC: Improve rice management',
       'Singleton - Default Value - AGRC: Decrease Exports',
       'Singleton - Default Value - AGRC: Reduce supply chain losses',
       'Singleton - Default Value - AGRC: Expand conservation agriculture',
       'Singleton - Default Value - AGRC: Improve crop productivity',
       'Singleton - Default Value - FRST: Increase Sequestration',
       'Singleton - Default Value - LNDU: Bound Classes',
       'Singleton - Default Value - LNDU: Decrease loss of land use classes',
       'Singleton - Default Value - LNDU: Stop deforestation',
       'Singleton - Default Value - LNDU: Expand sustainable grazing practices',
       'Singleton - Default Value - LNDU: Increase Reforestation',
       'Singleton - Default Value - LNDU: Expand silvopasture',
       'Singleton - Default Value - LNDU: Partial land use reallocation',
       'Singleton - Default Value - LSMM: Increase biogas capture at anaerobic decomposition facilitie

In [188]:
df_merged['transformation_name_sector'] = df_merged['transformation_name'] + " - " + df_merged['sector']

In [189]:
df_merged.to_csv(os.path.join(OUTPUT_DATA_DIR_PATH, "tornado_plot.csv"), index=False)

### Create a QA version

In [138]:
df_merged.head()

,strategy_id,primary_id,strategy,emission_total,base_emission_total,emission_diff,sector,transformation_name,air_pollution,congestion,...,road_safety,sector_specific,system_cost,technical_cost,technical_savings,water_pollution,net_benefit,additional_benefits,total_transformation_costs,marginal_total_abatement_cost_(USD/tCO2e)
0,1000.0,1001.0,Singleton - Default Value - AGRC: Improve rice...,2751.782000,2751.782,0.0,AGRC,Improve rice management,0.000000,0.0,...,0.0,0.000000,0.0,-0.000000,NaN,0.0,0.000000,0.000000,0.000000,NaN
1,1001.0,2002.0,Singleton - Default Value - AGRC: Decrease Exp...,2751.782000,2751.782,0.0,AGRC,Decrease Exports,0.000000,0.0,...,0.0,0.000000,0.0,-0.000000,NaN,0.0,0.000000,0.000000,0.000000,NaN
2,1002.0,3003.0,Singleton - Default Value - AGRC: Reduce suppl...,2751.666513,2751.782,-0.1,AGRC,Reduce supply chain losses,-0.000116,0.0,...,0.0,0.000474,0.0,-0.236728,3.570858,0.0,3.893872,3.657145,3.807346,-2367.278791
3,1003.0,4004.0,Singleton - Default Value - AGRC: Expand conse...,2753.552329,2751.782,1.8,AGRC,Expand conservation agriculture,0.000000,0.0,...,0.0,0.000000,0.0,-0.000000,2.971877,0.0,28.341555,28.341555,2.971877,0.000000
4,1004.0,5005.0,Singleton - Default Value - AGRC: Improve crop...,2757.219557,2751.782,5.4,AGRC,Improve crop productivity,-0.266912,0.0,...,0.0,-3.501832,0.0,0.201055,NaN,0.0,37.712410,37.913465,-0.201055,37.232435


In [139]:
df_merged.sector.unique()

array(['AGRC', 'FRST', 'LNDU', 'LSMM', 'LVST', 'SOIL', 'TRWW', 'WALI',
       'WASO', 'CCSQ', 'ENFU', 'ENTC', 'FGTV', 'INEN', 'SCOE', 'TRDE',
       'TRNS', 'IPPU', 'PFLO'], dtype=object)

In [140]:
relevant_fields = [
    "transformation_name",
    "sector",
    "base_emission_total",
    "emission_total",
    "emission_diff",
    "technical_cost",
    "marginal_total_abatement_cost_(USD/tCO2e)"
]

# keep only relevant fields
df_merged_filtered = df_merged[relevant_fields]
df_merged_filtered.head()

,transformation_name,sector,base_emission_total,emission_total,emission_diff,technical_cost,marginal_total_abatement_cost_(USD/tCO2e)
0,Improve rice management,AGRC,2751.782,2751.782000,0.0,-0.000000,NaN
1,Decrease Exports,AGRC,2751.782,2751.782000,0.0,-0.000000,NaN
2,Reduce supply chain losses,AGRC,2751.782,2751.666513,-0.1,-0.236728,-2367.278791
3,Expand conservation agriculture,AGRC,2751.782,2753.552329,1.8,-0.000000,0.000000
4,Improve crop productivity,AGRC,2751.782,2757.219557,5.4,0.201055,37.232435


In [141]:
df_merged_filtered

,transformation_name,sector,base_emission_total,emission_total,emission_diff,technical_cost,marginal_total_abatement_cost_(USD/tCO2e)
0,Improve rice management,AGRC,2751.782,2751.782000,0.0,-0.000000e+00,NaN
1,Decrease Exports,AGRC,2751.782,2751.782000,0.0,-0.000000e+00,NaN
2,Reduce supply chain losses,AGRC,2751.782,2751.666513,-0.1,-2.367279e-01,-2367.278791
3,Expand conservation agriculture,AGRC,2751.782,2753.552329,1.8,-0.000000e+00,0.000000
4,Improve crop productivity,AGRC,2751.782,2757.219557,5.4,2.010552e-01,37.232435
...,...,...,...,...,...,...,...
63,Reduce Nitrous Oxide emissions,IPPU,2751.782,2751.390786,-0.4,2.153358e-02,53.833952
64,Reduce other fluorinated compounds,IPPU,2751.782,2751.782000,0.0,-0.000000e+00,NaN
65,Reduce use of PFCs,IPPU,2751.782,2751.782000,0.0,4.434269e-08,inf
66,Change diets,PFLO,2751.782,2746.790351,-5.0,9.439921e-03,1.887984


In [142]:
df_merged_filtered.to_clipboard(index=False)

In [143]:
df_merged_filtered.to_csv(os.path.join(OUTPUT_DATA_DIR_PATH, "tornado_plot_for_QA.csv"), index=False)